<a href="https://colab.research.google.com/github/cetracey/ds2002-fa26/blob/main/Copy_of_2026_10_09_%E2%80%94_Public_API_to_Clean_Table_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [9]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

    delay = 1
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=10)

            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])

            if r.status_code in {400, 404}:
                r.raise_for_status()

            if r.status_code in RETRY_STATUSES:
                print(f"Warning: Status {r.status_code} received. Retrying in {delay} seconds...")
                time.sleep(delay)
                delay *= 2
                continue

            r.raise_for_status()

        except requests.exceptions.RequestException as e:
            if attempt == tries - 1:
                raise e
            print(f"Network error on attempt {attempt + 1}: {e}. Retrying in {delay} seconds...")
            time.sleep(delay)
            delay *= 2

    raise RuntimeError("All attempts to fetch weather data failed.")

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [10]:
used_fallback = False
try:
    raw_wx = get_weather(start='2026-09-05', end='2026-10-03')
    raw_wx = raw_wx[raw_wx['time'].isin(['2026-09-05', '2026-10-03'])].reset_index(drop=True)
except Exception as e:
    print(f"Failed to fetch from Open-Meteo API: {e}. Falling back to sample data.")
    used_fallback = True

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [11]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx = raw_wx.copy()
    wx['source'] = 'open-meteo'

display(wx)

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [12]:
wx['date'] = pd.to_datetime(wx['time'])
wx = wx.rename(columns={
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

wx = wx[['date', 'temp_max', 'precip_mm', 'source']]
display(wx)

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [13]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [14]:

sep_val = wx.loc[wx['date'] == '2026-09-05', 'precip_mm'].values[0]
oct_val = wx.loc[wx['date'] == '2026-10-03', 'precip_mm'].values[0]

difference = abs(oct_val - sep_val)

if oct_val > sep_val:
    print(f"October 3, 2026 was wetter than September 5, 2026 by {difference:.1f} mm of rain (October 3 had {oct_val:.1f} mm while September 5 had {sep_val:.1f} mm).")
elif sep_val > oct_val:
    print(f"September 5, 2026 was wetter than October 3, 2026 by {difference:.1f} mm of rain (September 5 had {sep_val:.1f} mm while October 3 had {oct_val:.1f} mm).")
else:
    print(f"Both dates had the same amount of rainfall ({sep_val:.1f} mm).")

October 3, 2026 was wetter than September 5, 2026 by 12.2 mm of rain (October 3 had 14.7 mm while September 5 had 2.5 mm).


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [15]:
import datetime

pull_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S NY Time")

print("=== Data Citation ===")
print(f"Endpoint: {BASE}")
print("Coordinates: Latitude 38.03, Longitude -78.51 (Charlottesville, VA)")
print("Date Range: 2026-09-05 to 2026-10-03")
print(f"Pulled on: {pull_time}")
print(f"Data Source Status: {wx['source'].iloc[0]}")

=== Data Citation ===
Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: Latitude 38.03, Longitude -78.51 (Charlottesville, VA)
Date Range: 2026-09-05 to 2026-10-03
Pulled on: 2026-10-11 17:46:53 NY Time
Data Source Status: open-meteo


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather returned an empty DataFrame on failure instead of raising, pandas would merge it with the 200 days of sales via a left/outer join. Since the weather dataframe is empty, the resulting columns (temp_max, precip_mm) would be filled entirely with NaN (or null values) for all 200 days.

When we plot rain against revenue, the plotting library (such as seaborn or matplotlib) typically drops rows containing NaN values silently when creating the chart. As a result, the chart would render as entirely blank (or with no data points plotted), but without throwing an error message. Nobody would catch it because there's no console failure, and users might assume there simply wasn't any matching data or that a styling parameter hid the points, rather than realizing the pipeline failed to fetch weather data entirely.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [16]:

cities_config = {
    'Charlottesville': {'lat': 38.03, 'lon': -78.51},
    'Orlando': {'lat': 28.54, 'lon': -81.38}
}

frames = []
for city_name, coords in cities_config.items():
    try:
        city_raw = get_weather(start='2026-09-05', end='2026-10-03', lat=coords['lat'], lon=coords['lon'])
        city_raw = city_raw[city_raw['time'].isin(['2026-09-05', '2026-10-03'])].reset_index(drop=True)
        city_df = city_raw.copy()
        city_df['source'] = 'open-meteo'
    except Exception as e:
        print(f"Failed to fetch for {city_name}, using fallback: {e}")

        if city_name == 'Charlottesville':
            city_df = FALLBACK.copy()
        else:
            city_df = pd.DataFrame({
                'time': ['2026-09-05', '2026-10-03'],
                'temperature_2m_max': [32.0, 29.0],
                'precipitation_sum': [5.0, 12.0],
            })
        city_df['source'] = 'fallback sample'

    city_df['date'] = pd.to_datetime(city_df['time'])
    city_df = city_df.rename(columns={
        'temperature_2m_max': 'temp_max',
        'precipitation_sum': 'precip_mm'
    })
    city_df['city'] = city_name
    city_df = city_df[['date', 'city', 'temp_max', 'precip_mm', 'source']]
    frames.append(city_df)

combined_wx = pd.concat(frames, ignore_index=True)
display(combined_wx)

,date,city,temp_max,precip_mm,source
0,2026-09-05,Charlottesville,29.4,2.5,open-meteo
1,2026-10-03,Charlottesville,22.0,14.7,open-meteo
2,2026-09-05,Orlando,32.1,3.9,open-meteo
3,2026-10-03,Orlando,30.4,14.5,open-meteo
